# Raw to Silver — cleaning and validation test

This notebook reads the MinIO raw layer and produces cleaned DataFrames **in memory only**. It intentionally does not write to `silver/` yet. Run every cell from top to bottom.

In [ ]:
import os

from dotenv import load_dotenv
from pyspark.sql import DataFrame, Window, functions as F

from src.spark_minio import get_spark_session

load_dotenv()
bucket_name = os.environ["MINIO_BUCKET"]
spark = get_spark_session("sanlam-raw-to-silver-test")
spark.sparkContext.setLogLevel("WARN")

In [ ]:
# Each raw table, its primary key, and any date/decimal columns that Silver must enforce.
TABLE_CONFIG = {
    "customers": {"pk": "customer_id", "dates": ["date_of_birth"], "decimals": {}},
    "agents": {"pk": "agent_id", "dates": [], "decimals": {}},
    "insurance_products": {"pk": "product_id", "dates": [], "decimals": {"base_monthly_premium": "decimal(12,2)", "max_cover_amount": "decimal(14,2)"}},
    "policies": {"pk": "policy_id", "dates": ["inception_date", "expiry_date", "cancellation_date"], "decimals": {"premium_amount": "decimal(12,2)", "sum_insured": "decimal(14,2)"}},
    "policy_coverages": {"pk": "policy_coverage_id", "dates": [], "decimals": {"coverage_limit": "decimal(14,2)", "deductible_amount": "decimal(12,2)"}},
    "policy_status_history": {"pk": "policy_status_history_id", "dates": ["changed_at"], "decimals": {}},
    "premium_transactions": {"pk": "premium_transaction_id", "dates": ["due_date", "transaction_date"], "decimals": {"amount": "decimal(12,2)"}},
    "claims": {"pk": "claim_id", "dates": ["loss_date", "reported_date", "closed_date"], "decimals": {"claimed_amount": "decimal(14,2)", "approved_amount": "decimal(14,2)"}},
    "claim_payments": {"pk": "claim_payment_id", "dates": ["payment_date"], "decimals": {"payment_amount": "decimal(14,2)"}},
}

ENUM_RULES = {
    "customers": {"customer_segment": ["MASS_MARKET", "AFFLUENT", "COMMERCIAL"]},
    "agents": {"active_flag": ["Y", "N"]},
    "insurance_products": {"active_flag": ["Y", "N"]},
    "policies": {"policy_status": ["PENDING", "ACTIVE", "LAPSED", "CANCELLED", "EXPIRED"]},
    "policy_coverages": {"active_flag": ["Y", "N"]},
    "premium_transactions": {"transaction_type": ["PREMIUM", "REVERSAL", "REFUND"], "transaction_status": ["PENDING", "PAID", "FAILED", "REVERSED"]},
    "claims": {"claim_status": ["OPEN", "UNDER_REVIEW", "APPROVED", "REJECTED", "SETTLED", "CLOSED"], "fraud_indicator": ["Y", "N"]},
    "claim_payments": {"payment_status": ["SCHEDULED", "PAID", "VOID"]},
}

In [ ]:
def read_raw_table(table_name: str) -> DataFrame:
    """Read one raw table and preserve the exact MinIO source file for tie-breaking."""
    return (
        spark.read.parquet(f"s3a://{bucket_name}/raw/{table_name.upper()}/")
        .withColumn("_source_file", F.input_file_name())
        #.withColumn adds a new column called _source_file to every row, containing the filename 
        # that row physically came from. We use it for the final tie brake to know which row to keep and which
        # to drop this makes us always keep data from the newest file
    )

def clean_table(raw_df: DataFrame, config: dict) -> DataFrame:
    """Standardize names, retain one latest record per key, and enforce Silver data types."""
    df = raw_df.toDF(*[name.lower() for name in raw_df.columns]) #renames all columns to become lowercase
    df = df.withColumn("load_date", F.to_date("load_date")) #date of when loaded, we use it as tie braker when dropping duplicates, if 2 rows have same last_updated.

    # LAST_UPDATED is the business version. The next two fields make timestamp ties repeatable.
    latest_first = Window.partitionBy(config["pk"]).orderBy(
        F.col("last_updated").desc(),
        F.col("load_date").desc_nulls_last(),
        F.col("_source_file").desc(),
    )
    df = (df.withColumn("_row_number", F.row_number().over(latest_first))
            .filter(F.col("_row_number") == 1)
            .drop("_row_number"))

    for column in config["dates"]:
        df = df.withColumn(column, F.col(column).cast("date"))
    for column, data_type in config["decimals"].items():
        df = df.withColumn(column, F.col(column).cast(data_type))
    return df

In [ ]:
raw_tables = {name: read_raw_table(name) for name in TABLE_CONFIG} #We apply previous fct on each table to read them
silver_test = {name: clean_table(raw_tables[name], config) for name, config in TABLE_CONFIG.items()} #We apply previous fct on each table to turn them to silver

# Table-specific, non-destructive cleaning rules.
silver_test["customers"] = silver_test["customers"].withColumn("has_email", F.col("email").isNotNull())
silver_test["policy_status_history"] = silver_test["policy_status_history"].withColumn(
    "old_status", F.coalesce(F.col("old_status"), F.lit("NEW_POLICY")) #for policy_status_history table if any value in the old_status column is null, we give a new value "NEW_POLICY", coalesce behaves like F.when fct
)

# Approved amount remains null until an approval decision exists. This keeps the source meaning intact.
silver_test["claims"] = silver_test["claims"].withColumn(
    "approved_amount",
    F.when(F.col("claim_status") == "REJECTED", F.lit(None).cast("decimal(14,2)")) 
     .otherwise(F.col("approved_amount")),
)#we always use lit to convert from a python value to a column object or expression so spark's API understands

In [ ]:
# USE OF FCT: after filtering a df based on condition, if any rows exist in this df after filtering
# we raise an error
def fail_if(condition_df: DataFrame, message: str) -> None:
    invalid_count = condition_df.count()    
    if invalid_count:
        raise ValueError(f"{message}: {invalid_count} invalid row(s)") 

# USE OF FCT : We take one dataframe and check if it verifies multiple necessary conditions
def validate_table(df: DataFrame, table_name: str, config: dict) -> None:
    pk = config["pk"]
    fail_if(df.filter(F.col(pk).isNull()), f"{table_name}.{pk} is null") #if primary key is null, we raise an error
    fail_if(df.groupBy(pk).count().filter(F.col("count") > 1), f"{table_name}.{pk} is duplicated") #if pk is duplicated, we raise an error
    for column, allowed in ENUM_RULES.get(table_name, {}).items(): #ENUM_RULES.get(table_name, {}).items() this code basically is equivalent to ENUM_RULES[table_name] but not safe, if that table_name doest exist in our dict it just crashes, instead we return an empty dict {} so the code doesnt crash, {} is called default value. because our orginial dict is composed of 2 dict inside eachother, we use table_name to get the second dict and do .items() to return both the key and value of that dict 
        fail_if(df.filter(F.col(column).isNotNull() & ~F.col(column).isin(allowed)),
                f"{table_name}.{column} is outside its allowed values") #simple line where we verify if a row is not null and contains a value of the values imposed by the oracle database when creating the table

for name, config in TABLE_CONFIG.items():
    validate_table(silver_test[name], name, config)

# Key relationships. A left-anti join returns child records that have no parent.
#it means we check if foreign key in the child table, exists in the parent table as primary key of course
def validate_relationship(child: str, fk: str, parent: str, pk: str) -> None:
    child_df = silver_test[child].filter(F.col(fk).isNotNull()).alias("child")
    parent_df = silver_test[parent].select(pk).alias("parent")
    orphans = child_df.join(
        parent_df,
        F.col(f"child.{fk}") == F.col(f"parent.{pk}"),
        "left_anti",
    )
    fail_if(orphans, f"{child}.{fk} has no parent in {parent}.{pk}")

validate_relationship("policies", "customer_id", "customers", "customer_id")
validate_relationship("policies", "product_id", "insurance_products", "product_id")
validate_relationship("policies", "agent_id", "agents", "agent_id")
validate_relationship("claims", "policy_id", "policies", "policy_id")
validate_relationship("claim_payments", "claim_id", "claims", "claim_id")
validate_relationship("premium_transactions", "policy_id", "policies", "policy_id")
validate_relationship("policy_coverages", "policy_id", "policies", "policy_id")
validate_relationship("policy_status_history", "policy_id", "policies", "policy_id")

In [ ]:
# Business and null rules. These validate; they do not silently change bad source data.
policies = silver_test["policies"]
claims = silver_test["claims"]
premium_transactions = silver_test["premium_transactions"]

fail_if(policies.filter(F.date_add("inception_date", 365) != F.col("expiry_date")), "policy expiry is not inception plus 365 days")
fail_if(policies.filter((F.col("policy_status") == "CANCELLED") & F.col("cancellation_date").isNull()), "cancelled policy has no cancellation_date")
fail_if(policies.filter((F.col("policy_status") != "CANCELLED") & F.col("cancellation_date").isNotNull()), "non-cancelled policy has cancellation_date")
fail_if(policies.filter((F.col("premium_amount") <= 0) | (F.col("sum_insured") <= 0)), "policy amount is not positive")

fail_if(claims.filter(F.col("loss_date") > F.col("reported_date")), "claim loss_date is after reported_date")
fail_if(claims.filter(F.col("approved_amount").isNotNull() & (F.col("approved_amount") > F.col("claimed_amount"))), "approved amount exceeds claimed amount")
fail_if(claims.filter(F.col("claim_status").isin("APPROVED", "SETTLED", "CLOSED") & F.col("approved_amount").isNull()), "decided claim has no approved amount")
fail_if(claims.filter(F.col("claim_status").isin("SETTLED", "CLOSED") & F.col("closed_date").isNull()), "closed claim has no closed_date")

fail_if(premium_transactions.filter((F.col("transaction_status") == "PAID") & F.col("transaction_date").isNull()), "paid premium has no transaction_date")
fail_if(premium_transactions.filter(F.col("amount") <= 0), "premium transaction amount is not positive")

print("All Silver test validations passed. No data has been written to MinIO silver/.")

In [ ]:
# Audit summary: compare raw record counts to the latest-record Silver-test counts.
for name in TABLE_CONFIG:
    print(f"{name:24} raw={raw_tables[name].count():6}  cleaned={silver_test[name].count():6}")

silver_test["customers"].printSchema()
silver_test["customers"].show(5, truncate=False)

## Publish to Silver

Run this cell only after every preceding validation cell has passed. It rebuilds the complete current-state Silver tables from all raw history.

In [ ]:
# Publish only after all validation cells above have completed successfully.
# `overwrite` is correct here: every Silver table is a newly rebuilt current-state snapshot.
SILVER_ROOT = f"s3a://{bucket_name}/silver"
PROCESSING_COLUMNS = ["_source_file"]

for table_name, dataframe in silver_test.items():
    output_path = f"{SILVER_ROOT}/{table_name}/"
    publish_df = dataframe.drop(*[
        column for column in PROCESSING_COLUMNS if column in dataframe.columns
    ])

    (
        publish_df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(output_path)
    )
    print(f"Published {table_name}: {publish_df.count()} rows -> {output_path}")

print("Silver publishing completed successfully.")